# 03 — Evaluate saved model & predict genres

This notebook:
1. Loads `models/genre_classifier.joblib`
2. Rebuilds the same **20% held-out test split** used in training (needs cleaned data / raw CSV)
3. Scores accuracy / F1 on the **9 parent genres**
4. Shows predicted vs true genres
5. Saves `outputs/test_predictions.csv`

**Canonical CLI:** `python scripts/evaluate_model.py`  
Committed metrics/predictions already reflect ~**59.1%** test accuracy.

Do **not** open the `.joblib` file in Jupyter’s file browser — load it with Python as below.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("..").resolve()

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

from src.data_io import AUDIO_FEATURES
from src.evaluate import evaluate_saved_model, predict_genres
from src.genre_map import PARENT_GENRES, parent_genre_guide_frame
from src.train import load_metrics, load_trained_model

display(parent_genre_guide_frame())

In [ ]:
model = load_trained_model()
print("Model classes:", list(model.classes_))
assert set(model.classes_) == set(PARENT_GENRES), "Unexpected class set vs PARENT_GENRES"

metrics = load_metrics()
print("Committed/saved accuracy:", round(metrics["accuracy"], 4))
print("Macro F1:", round(metrics["macro_f1"], 4))
print("n_classes:", metrics["n_classes"])

In [ ]:
result = evaluate_saved_model(n_examples=20, save_predictions=True)
result["accuracy"], result["macro_f1"], result["predictions_path"]

In [ ]:
preds = result["predictions"]
display(preds[["true_genre", "predicted_genre", "confidence", "correct"]].head(20))
print("Accuracy by true parent genre:")
display(
    preds.groupby("true_genre")["correct"]
    .mean()
    .sort_values(ascending=False)
    .rename("accuracy")
)

## Predict a single custom track
Edit the feature values below, then run the cell. The model returns one of the 9 parent genres.

In [ ]:
import pandas as pd

custom = pd.DataFrame([
    {
        "danceability": 0.72,
        "energy": 0.80,
        "key": 5,
        "loudness": -6.5,
        "mode": 1,
        "speechiness": 0.06,
        "acousticness": 0.12,
        "instrumentalness": 0.0,
        "liveness": 0.15,
        "valence": 0.65,
        "tempo": 118.0,
        "time_signature": 4,
        "duration_ms": 215000,
    }
])

predict_genres(custom)[["predicted_genre", "confidence", "top3_genres"]]